# DIT5411 Traditional Chinese Handwriting Recognition

This optional notebook runs the full 13,065-class workflow on a GPU-enabled Colab runtime. Local Apple Silicon execution is the primary workflow; Colab is only an alternative. Select **Runtime → Change runtime type → GPU** before continuing in Colab. Test images remain non-augmented and are used only for final evaluation.

## 1. Project setup

Clone the project and install its declared dependencies. If the project is already mounted, adjust `PROJECT_DIR` instead of cloning it again.

In [ ]:
from pathlib import Path
import os

PROJECT_DIR = Path('/content/DIT5411-FokHoYin')
if not PROJECT_DIR.exists():
    !git clone https://github.com/ShuoKuro/DIT5411-FokHoYin.git {PROJECT_DIR}
os.chdir(PROJECT_DIR)
%pip install -r requirements.txt

In [ ]:
import platform
import tensorflow as tf

print('Python:', platform.python_version())
print('TensorFlow:', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))
assert tf.config.list_physical_devices('GPU'), 'Enable a GPU runtime before full training.'

## 2. Full dataset

Download the full 13,065-class archive. Do not substitute the 4,803-class common-character dataset. The download and extraction steps require substantial time and free disk space.

In [ ]:
DATA_PARENT = Path('/content/dit5411_data')
SOURCE_DIR = DATA_PARENT / 'source/handwritting_data_all'
DATA_PARENT.mkdir(parents=True, exist_ok=True)
if not SOURCE_DIR.exists():
    !git clone https://github.com/chenkenanalytic/handwritting_data_all.git {SOURCE_DIR}
if not (DATA_PARENT / 'all_data.zip').exists():
    !cat {SOURCE_DIR}/all_data.zip* > {DATA_PARENT}/all_data.zip
if not (DATA_PARENT / 'cleaned_data').exists():
    !python -m src.extract_data --archive {DATA_PARENT}/all_data.zip --output-dir {DATA_PARENT}
DATA_DIR = DATA_PARENT / 'cleaned_data'
MANIFEST_DIR = PROJECT_DIR / 'outputs/data'
print('Dataset:', DATA_DIR)

## 3. Audit and split the data

Natural filename order determines the split. Standard classes use the first 40 images for training and all remaining images for testing. The 43 classes with only 16–40 images instead use approximately their first 80% for training while retaining original test images. Four training-pool images per class are reserved for development validation; the final full-pool run restores them to training. Exact counts are saved in the audit outputs.

In [ ]:
!python -m src.prepare_data --data-dir {DATA_DIR} --output-dir {MANIFEST_DIR} --verify-images

## 4. Inspect augmentation

Set `SAMPLE_IMAGE` to any image from the training pool. Rotation, shear, scaling, and translation are deterministic. Only training images are augmented.

In [ ]:
import csv

with (MANIFEST_DIR / 'train.csv').open(encoding='utf-8', newline='') as handle:
    first_training_row = next(csv.DictReader(handle))
SAMPLE_IMAGE = DATA_DIR / first_training_row['path']
!python -m src.augment --image {SAMPLE_IMAGE} --output outputs/augmentation_preview.png --seed 42

## 5. Train and compare three architectures

Each development run supplies exactly 200 samples per class per epoch. Keep the data and stopping settings unchanged for a fair comparison. Training the three full models requires a substantial GPU budget.

In [ ]:
!python -m src.train --data-dir {DATA_DIR} --model mlp --epochs 20 --learning-rate 0.0001 --mixed-precision --verbose 2

In [ ]:
!python -m src.train --data-dir {DATA_DIR} --model compact_cnn --epochs 20 --mixed-precision --verbose 2

In [ ]:
!python -m src.train --data-dir {DATA_DIR} --model deeper_cnn --epochs 20 --mixed-precision --verbose 2

## 6. Evaluate on original test images

Run all three evaluations. Use the measured Top-1 values in the README comparison table. Do not report validation accuracy as test accuracy.

In [ ]:
for model_name in ('mlp', 'compact_cnn', 'deeper_cnn'):
    model_path = PROJECT_DIR / f'outputs/runs/{model_name}/best_model.keras'
    !python -m src.evaluate --data-dir {DATA_DIR} --model-path {model_path}

## 7. Retrain the selected architecture with the complete training pool

Replace `deeper_cnn` and the epoch count below with the selected architecture and best development epoch count. This run uses fixed epochs rather than early stopping because validation images are returned to the training pool.

In [ ]:
!python -m src.train --data-dir {DATA_DIR} --model deeper_cnn --include-validation --epochs 20 --mixed-precision --verbose 2 --output-dir outputs/runs/deeper_cnn_full40
!python -m src.evaluate --data-dir {DATA_DIR} --model-path outputs/runs/deeper_cnn_full40/final_model.keras

## 8. Single-image inference

Use an original handwritten image. The output contains the predicted Chinese character and ranked probabilities.

In [ ]:
TEST_IMAGE = SAMPLE_IMAGE
!python -m src.predict --model-path outputs/runs/deeper_cnn_full40/final_model.keras --image {TEST_IMAGE} --manifest-dir {MANIFEST_DIR}

## 9. Finalize the submission

Copy only measured metrics into the English README. Include all model descriptions, test Top-1 and Top-5 accuracies, the selected model, augmentation details, software versions, hardware, and references. Keep raw data, checkpoints, and large output files out of Git.